To demonstrate the ``CPP().simplify()`` method, we load the ``DOM_GSEC`` example dataset (see [Breimann25a]_) and obtain a feature set with ``CPP().run()`` using an interpretability-tiered scale set (``load_scales(top_explain_n=...)``). ``simplify`` rewrites this feature set into a more interpretable, and ideally smaller, one: for each feature it swaps the scale for a correlated scale from a better-rated AAontology subcategory (interpretability rating 1-10, 1 = best), recomputes the feature statistics, accepts the swap only behind CPP filtering and a cross-validation gate, and finally removes redundant features. It returns the simplified ``df_feat`` (and, optionally, a long-form table of every candidate considered).

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False
df_seq = aa.load_dataset(name="DOM_GSEC", n=25)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
df_scales = aa.load_scales(top_explain_n=20)
cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales)
df_feat = cpp.run(labels=labels, n_filter=12)
aa.display_df(df_feat, n_rows=5, show_shape=True)

DataFrame shape: (12, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,12)-BEGF750101",Conformation,α-helix,α-helix,"Conformational ...in-Dirkx, 1975)",0.419000,0.225000,0.225000,0.071000,0.169000,0.000000,0.002796,"24,28,32"
3,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
4,"TMD-Pattern(C,4...,11)-BEGF750101",Conformation,α-helix,α-helix,"Conformational ...in-Dirkx, 1975)",0.396000,0.238000,0.238000,0.106000,0.168000,0.000002,0.002369,"20,23,27"
5,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"


With only ``df_feat`` and ``labels`` (and a ``random_state`` for reproducibility), ``simplify`` runs the default ``greedy`` strategy with an SVM cross-validation gate and returns the simplified feature set:

In [2]:
df_simple = cpp.simplify(df_feat=df_feat, labels=labels, random_state=0)
aa.display_df(df_simple, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


**Target selection** chooses which features to attempt swapping. ``top_n`` targets the n worst-interpretability features, while ``max_interpretability`` instead targets every feature whose scale subcategory is rated worse (higher) than a ceiling. The two are mutually exclusive (set at most one):

In [3]:
# top_n: target the 5 worst-interpretability features
df_top = cpp.simplify(df_feat=df_feat, labels=labels, top_n=5, random_state=0)
# max_interpretability: alternatively, target everything rated worse than tier 3
df_ceil = cpp.simplify(df_feat=df_feat, labels=labels, max_interpretability=3, random_state=0)
aa.display_df(df_top, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


The **strategy** controls how swaps are chosen and validated. ``greedy`` swaps feature by feature behind the CV gate; ``consolidate`` batches features by subcategory toward the fewest subcategories; ``swap_all`` applies every eligible swap with no cross-validation (fastest). The comparison below shows how each affects the feature and subcategory counts:

In [4]:
import pandas as pd
rows = []
for strategy in ["greedy", "consolidate", "swap_all"]:
    out = cpp.simplify(df_feat=df_feat, labels=labels, top_n=8, strategy=strategy,
                       random_state=0)
    rows.append([strategy, len(out), out["subcategory"].nunique()])
df_strategies = pd.DataFrame(rows, columns=["strategy", "n_features", "n_subcategories"])
aa.display_df(df_strategies)

,strategy,n_features,n_subcategories
1,greedy,9,5
2,consolidate,9,5
3,swap_all,9,5


The **cross-validation gate** (used by ``greedy`` and ``consolidate``) decides whether a swap is kept. ``ml_model`` selects the classifier as a preset ``'svm'`` (default), ``'rf'`` (recommended for non-linear relationships, slower), or ``'log_reg'``, or a custom scikit-learn estimator instance; ``metric`` is the scoring metric; ``n_cv`` is the number of folds; and ``tol`` is how much performance drop is tolerated (a swap is kept if its CV score is at least ``baseline - tol``):

In [5]:
df_rf = cpp.simplify(df_feat=df_feat, labels=labels, top_n=5,
                     ml_model="rf", metric="accuracy", n_cv=3, tol=0.05, random_state=0)
aa.display_df(df_rf, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


A custom estimator instance can be passed directly as ``ml_model`` (used as-is):

In [6]:
from sklearn.svm import SVC
df_custom = cpp.simplify(df_feat=df_feat, labels=labels, top_n=5,
                         ml_model=SVC(kernel="linear"), random_state=0)
aa.display_df(df_custom, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


**Candidate eligibility and per-feature filtering.** ``min_cor`` is the minimum absolute correlation a candidate scale must have with the original scale (anti-correlation allowed), and ``max_std_test`` is the CPP per-feature pre-filter threshold the recomputed swapped feature must satisfy:

In [7]:
df_strict = cpp.simplify(df_feat=df_feat, labels=labels, top_n=5,
                         min_cor=0.8, max_std_test=0.2, random_state=0)
aa.display_df(df_strict, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


**Unimprovable features.** When a targeted feature has no accepted swap, ``on_unimprovable`` decides its fate: ``'keep'`` (retain the original, default), ``'drop'`` (remove it), or ``'drop_if_perf_allows'`` (remove only if the CV score does not drop). The last feature is never dropped:

In [8]:
df_drop = cpp.simplify(df_feat=df_feat, labels=labels, top_n=5,
                       on_unimprovable="drop", random_state=0)
aa.display_df(df_drop, show_shape=True)

DataFrame shape: (7, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
2,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
3,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
4,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
5,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
6,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
7,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


**Redundancy reduction** removes features whose swapped scales became redundant. ``redundancy_tie_break`` chooses the keeper of a redundant pair (``'interpretability'`` or ``'performance'``); ``max_cor`` and ``max_overlap`` are the scale-correlation and position-overlap thresholds; and ``check_cat`` restricts comparisons to the same scale category:

In [9]:
df_red = cpp.simplify(df_feat=df_feat, labels=labels, top_n=8,
                      redundancy_tie_break="performance", max_cor=0.5, max_overlap=0.5,
                      check_cat=True, random_state=0)
aa.display_df(df_red, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


**Labels and reproducibility.** ``label_test`` / ``label_ref`` name the test and reference classes in ``labels`` (default 1 / 0), and ``random_state`` seeds the CV-gate model so the result is reproducible. (The optional ``X`` argument accepts a precomputed feature matrix as the baseline; when ``None`` it is recomputed internally, and swapped columns are always recomputed.)

In [10]:
df_repro = cpp.simplify(df_feat=df_feat, labels=labels,
                        label_test=1, label_ref=0, top_n=5, random_state=42)
aa.display_df(df_repro, show_shape=True)

DataFrame shape: (9, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_C_JMD_C-Pat...,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.426000,0.266000,-0.266000,0.097000,0.151000,0.000000,0.003705,"24,28,32"
2,"TMD_C_JMD_C-Pat...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.408000,0.312000,-0.312000,0.137000,0.172000,0.000001,0.003779,"27,31"
3,"TMD_C_JMD_C-Seg...5,7)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.385000,0.238000,0.238000,0.162000,0.144000,0.000003,0.002892,"32,33,34"
4,"TMD_C_JMD_C-Seg...,15)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002667,33
5,"TMD_C_JMD_C-Seg...,10)-DAWD720101",ASA/Volume,Volume,Volume,"Size (Dawson, 1972)",0.376000,0.263000,0.263000,0.151000,0.219000,0.000005,0.002495,"33,34"
6,"TMD-Segment(11,12)-GUYH850105",ASA/Volume,Accessible surface area (ASA),Partition energy,"Apparent partit...dex (Guy, 1985)",0.372000,0.127000,-0.127000,0.058000,0.132000,0.000006,0.002163,"27,28"
7,"TMD-Pattern(C,4...,11)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001670,"20,23,27"
8,"TMD_C_JMD_C-Pat...4,8)-FUKS010110",Composition,AA composition,Proteins of mesophiles (INT),"Entire chain co...ishikawa, 2001)",0.371000,0.206000,0.206000,0.151000,0.131000,0.000007,0.001698,"21,24,28"
9,"TMD_C_JMD_C-Seg...6,9)-VHEG790101",ASA/Volume,Accessible surface area (ASA),TFE to lipophilic phase,"Transfer free e...Blomberg, 1979)",0.366000,0.259000,0.259000,0.183000,0.141000,0.000009,0.001841,"32,33"


Finally, ``return_details=True`` additionally returns a long-form table of every candidate scale considered for each feature, with its interpretability rating, correlation with the original scale, recomputed ``std_test``, and whether it was accepted:

In [11]:
df_simple2, df_candidates = cpp.simplify(df_feat=df_feat, labels=labels, top_n=8,
                                         return_details=True, random_state=0)
aa.display_df(df_candidates, show_shape=True)

DataFrame shape: (1, 9)


,feature,candidate_scale,interpretability_orig,interpretability_cand,cor,std_test,accepted,cv_score,reason
1,"TMD_C_JMD_C-Seg...5,7)-OOBM770102",KARS160107,4.000000,1.000000,0.876388,0.138989,True,0.920000,accepted
